In [1]:
import numpy as np

from utils import symmetrized_pairs
from ERS_num_counts_xcls import (
    tucci_cross_spectra, 
    tucci_number_counts
)
from ERS_number_counts_5GHz import (
    TotalDiffNumCounts5GHz,
    TucciFlatInvDiffNumCounts5GHz
)
from fsky_loss import compute_fsky_PS_mask

In [2]:
S_lim = 1.0  # Jy   Flux cut for PS mask
nu_for_S_lim = 27.0    # GHz   Frequency band at which PS masking is determined
f_fwhm = 1.   # fraction of FWHM to use for PS mask

In [3]:
# example values for fsky of binary and weighted masks
fsky_binary_mask = 0.340
fsky = 0.097

In [4]:
instr_frequency = np.array([27., 39., 93., 145., 225., 280.])
instr_depth_p = np.array([49.5, 29.7, 3.7, 4.7, 8.9, 22.6])
instr_fwhm =np.array([91., 63., 30., 17., 11., 9.])
n_freq = len(instr_frequency)

In [ ]:
# load digitized Toffolatti total 5 GHz differential n counts from Fig. 1 of Toffolatti et al. (1998)
total_diff_ncounts = TotalDiffNumCounts5GHz.from_csv(
    "/data/tot_ncounts_from_toffolatti98_fig1_solid_line.csv"
)

# flat+inv differential n counts at 5 GHz
flatinv_diff_ncounts = TucciFlatInvDiffNumCounts5GHz()

In [6]:
# Flux grid at 5GHz
S5_min = 1e-05  # Jy
S5_max = 100    # Jy
ngrid = 10**3
S5_grid = np.logspace(np.log10(S5_min), np.log10(S5_max), ngrid)

In [7]:
freq_pairs_to_evaluate = instr_frequency[symmetrized_pairs(n_freq)]

In [8]:
instr_fwhm_dict = {instr_frequency[i]: instr_fwhm[i] for i in range(len(instr_frequency))}

In [9]:
# spectra_SO_inv = tucci_cross_spectra(
#     nu_pairs_GHz=freq_pairs_to_evaluate,
#     S5_grid_jy=S5_grid,
#     dNdS_total_5GHz=total_diff_ncounts.dNdS,
#     dNdS_flatinv_5GHz=flatinv_diff_ncounts.dNdS,
#     mask_nu_GHz=nu_for_S_lim,
#     mask_Slim_jy=S_lim,
#     mask_f_fwhm=f_fwhm,  # fraction of FWHM to use for PS mask
#     mask_beam_fwhm_arcmin=instr_fwhm_dict[nu_for_S_lim],   # beam FWHM at 27 GHz
#     point_source_fwhm_arcmin=instr_fwhm_dict,    # beams of the different bands
#     population='inverted',
#     verbose = True
# )

spectra_SO_steep = tucci_cross_spectra(
    nu_pairs_GHz=freq_pairs_to_evaluate,
    S5_grid_jy=S5_grid,
    dNdS_total_5GHz=total_diff_ncounts.dNdS,
    dNdS_flatinv_5GHz=flatinv_diff_ncounts.dNdS,
    mask_nu_GHz=nu_for_S_lim,
    mask_Slim_jy=S_lim,
    mask_f_fwhm=f_fwhm,  # fraction of FWHM to use for PS mask
    mask_beam_fwhm_arcmin=instr_fwhm_dict[nu_for_S_lim],   # beam FWHM at 27 GHz
    point_source_fwhm_arcmin=instr_fwhm_dict,    # beams of the different bands
    population='steep',
)

spectra_SO_flat = tucci_cross_spectra(
    nu_pairs_GHz=freq_pairs_to_evaluate,
    S5_grid_jy=S5_grid,
    dNdS_total_5GHz=total_diff_ncounts.dNdS,
    dNdS_flatinv_5GHz=flatinv_diff_ncounts.dNdS,
    mask_nu_GHz=nu_for_S_lim,
    mask_Slim_jy=S_lim,
    mask_f_fwhm=f_fwhm,  # fraction of FWHM to use for PS mask
    mask_beam_fwhm_arcmin=instr_fwhm_dict[nu_for_S_lim],   # beam FWHM at 27 GHz
    point_source_fwhm_arcmin=instr_fwhm_dict,    # beams of the different bands
    population='flat',
)

all_dicts = {
    "spectra_SO_steep": spectra_SO_steep,
    "spectra_SO_flat": spectra_SO_flat,
    # "spectra_SO_inv": spectra_SO_inv,
}

In [10]:
print(spectra_SO_steep)

{(27.0, 27.0): np.float64(6.382490677718795e-06), (27.0, 39.0): np.float64(1.7596538666654128e-06), (27.0, 93.0): np.float64(1.274173188552468e-07), (27.0, 145.0): np.float64(4.73103886807241e-08), (27.0, 225.0): np.float64(2.5756464202277478e-08), (27.0, 280.0): np.float64(2.404042855069971e-08), (39.0, 39.0): np.float64(5.361794128790646e-07), (39.0, 93.0): np.float64(4.4741129207367425e-08), (39.0, 145.0): np.float64(1.6784063759202923e-08), (39.0, 225.0): np.float64(9.219793389684933e-09), (39.0, 280.0): np.float64(8.643878915502034e-09), (93.0, 93.0): np.float64(4.439735948495993e-09), (93.0, 145.0): np.float64(1.703353541364099e-09), (93.0, 225.0): np.float64(9.555870402761744e-10), (93.0, 280.0): np.float64(9.052963023053782e-10), (145.0, 145.0): np.float64(6.606436287318308e-10), (145.0, 225.0): np.float64(3.746013129014738e-10), (145.0, 280.0): np.float64(3.5677009583448675e-10), (225.0, 225.0): np.float64(2.146513399514529e-10), (225.0, 280.0): np.float64(2.0550112682222738e-

In [11]:
N_steep_27 = tucci_number_counts(
    nu_GHz=nu_for_S_lim,
    S_lim_jy=S_lim,
    S5_grid_jy=S5_grid,
    dNdS_total_5GHz=total_diff_ncounts.dNdS,
    dNdS_flatinv_5GHz=flatinv_diff_ncounts.dNdS,
    population="steep",
)

N_flat_27 = tucci_number_counts(
    nu_GHz=nu_for_S_lim,
    S_lim_jy=S_lim,
    S5_grid_jy=S5_grid,
    dNdS_total_5GHz=total_diff_ncounts.dNdS,
    dNdS_flatinv_5GHz=flatinv_diff_ncounts.dNdS,
    population="flat",
)

# N_inv_27 = tucci_number_counts(
#     nu_GHz=nu_for_S_lim,
#     S_lim_jy=S_lim,
#     S5_grid_jy=S5_grid,
#     dNdS_total_5GHz=total_diff_ncounts.dNdS,
#     dNdS_flatinv_5GHz=flatinv_diff_ncounts.dNdS,
#     population="inverted",
# )

In [12]:
print(f"N_steep_27 = {N_steep_27 * 4 * np.pi * fsky_binary_mask:.3e}")
print(f"N_flat_27 = {N_flat_27 * 4 * np.pi * fsky_binary_mask:.3e}")

N_PS = (N_steep_27 + N_flat_27) * 4 * np.pi * fsky_binary_mask  # convert from sr^-1 to total number in the sky aread observed
area_masked, fsky_loss, percentage_of_fsky_lost, fsky_remaining_after_applying_PS_mask = compute_fsky_PS_mask(fsky_init_binary=fsky_binary_mask, fsky_init_weighted=fsky, N_PS=N_PS, FWHM=instr_fwhm[np.where(instr_frequency == nu_for_S_lim)][0], f_fwhm=f_fwhm)

print(f"Number of PS masked = {N_PS:.3e}")
print(f"Area masked by PS masks: {area_masked:.6f} steradians")
print(f"fsky loss due to PS masks: {fsky_loss:.6f}")
print(f"Percentage of fsky lost: {percentage_of_fsky_lost:.6f}")
print(f"fsky remaining after applying PS masks: {fsky_remaining_after_applying_PS_mask:.10f}")

N_steep_27 = 5.661e+00
N_flat_27 = 6.154e+01
Number of PS masked = 6.720e+01
Area masked by PS masks: 0.010505 steradians
fsky loss due to PS masks: 0.000836
Percentage of fsky lost: 0.008618
fsky remaining after applying PS masks: 0.0961640414
